In [ ]:
!pip install -q transformers accelerate peft bitsandbytes fastapi uvicorn pyngrok

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded successfully!")

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
def generate_response(user_message: str, system_prompt: str = "You are a helpful assistant.") -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    generated_ids = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Qwen2 API")

class PromptRequest(BaseModel):
    text: str
    system_prompt: Optional[str] = "You are a helpful assistant."

@app.post("/generate")
def generate(req: PromptRequest):
    result = generate_response(req.text, req.system_prompt)
    return {"response": result}

In [ ]:
import threading
import uvicorn

thread = threading.Thread(target=uvicorn.run, args=(app,), kwargs={"host": "0.0.0.0", "port": 8000})
thread.daemon = True
thread.start()

print("FastAPI server running on port 8000")

In [ ]:
from pyngrok import ngrok
import time

time.sleep(2)  # wait for server to start

public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Generate endpoint: {public_url}/generate")
print()
print("Example usage:")
print(f'curl -X POST {public_url}/generate -H "Content-Type: application/json" -d \'{{"text": "Hello, who are you?"}}\'')